# 📊 Tahap 1 — Data Understanding

Tujuan: Kenali dataset sebelum diproses lebih lanjut.

Dataset: **International Football Results** (1872–2024)

Pertanyaan yang ingin kita jawab:
- Berapa banyak data yang kita punya?
- Kolom apa saja yang tersedia?
- Ada data yang hilang (missing values) tidak?
- Bagaimana distribusi data secara umum?

### Setup untuk Google Colab

Jalankan cell di bawah ini dulu jika kamu membuka notebook ini lewat **Google Colab**.

Kalau dijalankan secara lokal (Jupyter Notebook di laptop), cell ini otomatis dilewati.

In [ ]:
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    if not os.path.exists('worldcup-win-predicition'):
        get_ipython().system('git clone https://github.com/vnymyz/worldcup-win-predicition.git')
    os.chdir('worldcup-win-predicition/notebooks')
    print('Setup Colab selesai! Folder kerja:', os.getcwd())
else:
    print('Berjalan secara lokal -- tidak perlu clone repo.')


## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Supaya grafik tampil di dalam notebook
%matplotlib inline

# Supaya output lebih rapi
pd.set_option('display.max_columns', None)

print('Library berhasil di-import! ✅')

Library berhasil di-import! ✅


## 2. Load Dataset

In [2]:
df = pd.read_csv('../data/raw/results.csv')

print(f'Dataset berhasil dimuat!')
print(f'Jumlah baris : {df.shape[0]:,}')
print(f'Jumlah kolom : {df.shape[1]}')

Dataset berhasil dimuat!
Jumlah baris : 49,477
Jumlah kolom : 9


## 3. Lihat Data Awal

`head()` menampilkan 5 baris pertama — untuk memastikan data terbaca dengan benar.

In [3]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [4]:
# Lihat 5 baris terakhir
df.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49476,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


## 4. Informasi Kolom

`info()` menampilkan nama kolom, tipe data, dan apakah ada missing values.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  object 
 1   home_team   49477 non-null  object 
 2   away_team   49477 non-null  object 
 3   home_score  49421 non-null  float64
 4   away_score  49421 non-null  float64
 5   tournament  49477 non-null  object 
 6   city        49477 non-null  object 
 7   country     49477 non-null  object 
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), object(6)
memory usage: 3.1+ MB


**Penjelasan kolom:**

| Kolom | Keterangan |
|---|---|
| `date` | Tanggal pertandingan |
| `home_team` | Tim tuan rumah |
| `away_team` | Tim tamu |
| `home_score` | Gol tim tuan rumah |
| `away_score` | Gol tim tamu |
| `tournament` | Nama turnamen (Friendly, FIFA World Cup, dll) |
| `city` | Kota pertandingan |
| `country` | Negara pertandingan |
| `neutral` | True = lapangan netral, False = kandang salah satu tim |

## 5. Cek Missing Values

In [6]:
missing = df.isnull().sum()
print('Missing values per kolom:')
print(missing)

Missing values per kolom:
date           0
home_team      0
away_team      0
home_score    56
away_score    56
tournament     0
city           0
country        0
neutral        0
dtype: int64


In [7]:
# Persentase missing values
missing_pct = (df.isnull().sum() / len(df)) * 100
print('Persentase missing values:')
print(missing_pct.round(2))

Persentase missing values:
date          0.00
home_team     0.00
away_team     0.00
home_score    0.11
away_score    0.11
tournament    0.00
city          0.00
country       0.00
neutral       0.00
dtype: float64


## 6. Statistik Dasar

In [8]:
df.describe()

,home_score,away_score
count,49421.000000,49421.000000
mean,1.757330,1.181826
std,1.774309,1.401899
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,31.000000,21.000000


## 7. Eksplorasi Awal — Jenis Turnamen

Kita perlu tahu ada turnamen apa saja, karena nanti kita akan **filter hanya World Cup**.

In [9]:
print(f'Total jenis turnamen: {df["tournament"].nunique()}')
print()
print('10 turnamen terbanyak:')
print(df['tournament'].value_counts().head(10))

Total jenis turnamen: 200

10 turnamen terbanyak:
tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64


In [10]:
# Cek nama persis turnamen World Cup
wc_tournaments = df[df['tournament'].str.contains('World Cup', case=False, na=False)]['tournament'].unique()
print('Turnamen yang mengandung kata "World Cup":')
print(wc_tournaments)

Turnamen yang mengandung kata "World Cup":
['FIFA World Cup' 'FIFA World Cup qualification' 'Viva World Cup'
 'CONIFA World Cup qualification']


## 8. Preview Data World Cup

Filter sementara untuk melihat seperti apa data World Cup.

In [11]:
df_wc = df[df['tournament'] == 'FIFA World Cup'].copy()

print(f'Jumlah pertandingan World Cup: {len(df_wc):,}')
print(f'Tahun pertama: {df_wc["date"].min()}')
print(f'Tahun terakhir: {df_wc["date"].max()}')
print()
df_wc.head(10)

Jumlah pertandingan World Cup: 1,036
Tahun pertama: 1930-07-13
Tahun terakhir: 2026-06-27



,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
1490,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1491,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True
1492,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True
1493,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1494,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True
1495,1930-07-16,Chile,Mexico,3.0,0.0,FIFA World Cup,Montevideo,Uruguay,True
1496,1930-07-17,Bolivia,Yugoslavia,0.0,4.0,FIFA World Cup,Montevideo,Uruguay,True
1497,1930-07-17,Paraguay,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1499,1930-07-18,Uruguay,Peru,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,False
1500,1930-07-19,Argentina,Mexico,6.0,3.0,FIFA World Cup,Montevideo,Uruguay,True


## 9. Berapa Tim yang Pernah Ikut World Cup?

In [12]:
all_teams = sorted(pd.concat([df_wc['home_team'], df_wc['away_team']]).unique())
print(f'Total tim yang pernah ikut World Cup: {len(all_teams)}')
print()

# Tabel rapi dengan nomor urut
df_teams = pd.DataFrame(
    [(i + 1, team) for i, team in enumerate(all_teams)],
    columns=['No', 'Nama Tim']
)

# Tampilkan semua baris tanpa terpotong
pd.set_option('display.max_rows', None)
df_teams.style.set_properties(**{'text-align': 'left'}).hide(axis='index')

Total tim yang pernah ikut World Cup: 86



No,Nama Tim
1,Algeria
2,Angola
3,Argentina
4,Australia
5,Austria
6,Belgium
7,Bolivia
8,Bosnia and Herzegovina
9,Brazil
10,Bulgaria


## ✅ Ringkasan Data Understanding

Isi setelah eksplorasi selesai:

- **Total data:** ... baris, ... kolom
- **Rentang waktu:** 1872 – 2024
- **Data World Cup:** ... pertandingan
- **Missing values:** ada / tidak ada di kolom ...
- **Observasi menarik:** ...

➡️ Lanjut ke **Notebook 02 — Data Cleaning**